# Parking-lot YOLOv11 — training results

Plots `runs/detect/parking_yolo11s/results.csv` from the 100-epoch run.

Use the **`.venv` kernel** in Jupyter/VS Code, or run the code with `.venv/bin/python`. Only `matplotlib` is needed (plus the standard library).

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt

CSV = Path('runs/detect/parking_yolo11s/results.csv')
if not CSV.exists():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / CSV
        if candidate.exists():
            CSV = candidate
            break
print('Reading:', CSV)

In [ ]:
with CSV.open(newline='') as f:
    rows = [{k: float(v) for k, v in row.items()} for row in csv.DictReader(f)]

epochs = [r['epoch'] for r in rows]
columns = [c for c in rows[0] if c != 'epoch']

def series(name):
    return [r[name] for r in rows]

print(f'{len(rows)} epochs | {len(columns)} columns | first={epochs[0]:.0f} last={epochs[-1]:.0f}')
print(f'Total wall time: {rows[-1]["time"] / 60:.1f} min')

In [ ]:
def plot_lines(specs, title, ylabel, figsize=(11, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    for col, label in specs:
        ax.plot(epochs, series(col), label=label, linewidth=1.8)
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.set_ylabel(ylabel)
    ax.set_xlim(epochs[0], epochs[-1])
    ax.grid(alpha=0.3)
    ax.legend(ncol=4, fontsize=8)
    fig.tight_layout()
    return fig, ax

## Loss curves — train vs validation

In [ ]:
fig_loss, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for j, kind in enumerate(('box_loss', 'cls_loss', 'dfl_loss')):
    for i, split in enumerate(('train', 'val')):
        col = f'{split}/{kind}'
        axes[i, j].plot(epochs, series(col),
                        color='tab:blue' if split == 'train' else 'tab:orange',
                        linewidth=1.6)
        axes[i, j].set_title(col)
        axes[i, j].set_ylabel('loss')
        axes[i, j].set_xlabel('epoch' if i == 1 else '')
        axes[i, j].grid(alpha=0.3)
fig_loss.suptitle('Training (top) vs validation (bottom) losses')
fig_loss.tight_layout()

## Validation metrics

In [ ]:
fig_metrics, ax_metrics = plot_lines([
    ('metrics/precision(B)', 'precision'),
    ('metrics/recall(B)', 'recall'),
    ('metrics/mAP50(B)', 'mAP50'),
    ('metrics/mAP50-95(B)', 'mAP50-95'),
], 'Validation metrics', 'score', figsize=(11, 5))

In [ ]:
map_key = 'metrics/mAP50-95(B)'
best_i = max(range(len(rows)), key=lambda i: rows[i][map_key])

fig_map, ax_map = plot_lines([(map_key, 'mAP50-95')], 'mAP50-95 with best epoch', 'score', figsize=(10, 4))
ax_map.scatter([epochs[best_i]], [rows[best_i][map_key]], color='red', zorder=5,
               label=f'best ep {int(epochs[best_i])}: {rows[best_i][map_key]:.4f}')
ax_map.legend(fontsize=8)

## Learning-rate schedule

In [ ]:
lr_cols = [c for c in columns if c.startswith('lr/')]
fig_lr, ax_lr = plot_lines([(c, c) for c in lr_cols], 'Learning rate per parameter group', 'lr', figsize=(10, 4))
ax_lr.set_yscale('log')

## Summary

In [ ]:
metric_keys = ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', map_key]
print(f'Best epoch by mAP50-95: {int(epochs[best_i])}')
for key in metric_keys:
    values = series(key)
    peak = max(values)
    print(f'{key:<28} best={peak:.4f} (ep {int(epochs[values.index(peak)])})  last={values[-1]:.4f}')
print(f'Total wall time: {rows[-1]["time"] / 60:.1f} min')

## Save the figures as PNG next to `results.csv`

In [ ]:
out_dir = CSV.parent / 'analysis_plots'
out_dir.mkdir(exist_ok=True)
for name, figure in [('losses', fig_loss), ('metrics', fig_metrics),
                     ('map50_95', fig_map), ('learning_rate', fig_lr)]:
    figure.savefig(out_dir / f'{name}.png', dpi=150)
print('Saved:', ', '.join(p.name for p in sorted(out_dir.glob('*.png'))))